# Delta tables in the Polaris catalog

In [04a](04a_write_delta.ipynb) we reached a Delta table by its path. Here we also register it in **Apache Polaris**, so it has a name: `lakehouse.delta_db.mall_customers`.

Polaris treats Delta as a *generic table*. It stores the table's name, format and location. The table's history stays in `_delta_log`, so Polaris does not move a pointer on each commit as it does for Iceberg.

**Before you start:** `polaris` and `rustfs` must be running, and `Mall_Customers.csv` must be in `/opt/examples/datasets`.

**Versions:** Polaris has no Spark plugin for 4.1 yet, so we use the 4.0 plugin. It works here with Delta **4.2.0**. Delta 4.3.1 from notebook 04a crashes on `CREATE TABLE` through this plugin.

**Limits of Delta in Polaris:** `saveAsTable` is not supported, and `CREATE TABLE` needs an explicit `LOCATION`. So we write the files first, then register them.

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("Delta Polaris")
    .master("local[2]")
    .config("spark.jars.packages", "org.apache.polaris:polaris-spark-4.0_2.13:1.8.0,org.apache.iceberg:iceberg-aws-bundle:1.12.0,io.delta:delta-spark_2.13:4.2.0")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Catalog named lakehouse, served by Polaris
    .config("spark.sql.catalog.lakehouse", "org.apache.polaris.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.uri", "http://polaris:8181/api/catalog")
    .config("spark.sql.catalog.lakehouse.warehouse", "lakehouse")
    .config("spark.sql.catalog.lakehouse.credential", "root:s3cr3t")
    .config("spark.sql.catalog.lakehouse.scope", "PRINCIPAL_ROLE:ALL")
    .config("spark.sql.sources.useV1SourceList", "")
    # Delta reaches RustFS through s3a://
    .config("spark.hadoop.fs.s3a.endpoint", "http://rustfs:9000")
    .config("spark.hadoop.fs.s3a.access.key", "dataops")
    .config("spark.hadoop.fs.s3a.secret.key", "Ankara06")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("file:///opt/examples/datasets/Mall_Customers.csv")

df.printSchema()

## 1. Write the files, then register the table

In [ ]:
delta_path = "s3a://warehouse/delta_db/mall_customers"

df.write.format("delta").mode("overwrite").save(delta_path)

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.delta_db")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS lakehouse.delta_db.mall_customers
    USING delta
    LOCATION '{delta_path}'
""")

In [ ]:
spark.sql("SHOW TABLES IN lakehouse.delta_db").show()

## 2. Read it by name

In [ ]:
spark.table("lakehouse.delta_db.mall_customers").count()

## 3. What does the catalog store?

In [ ]:
import psycopg2

def show_catalog(table):
    conn = psycopg2.connect(host="postgresql", user="postgres", password="Ankara06", dbname="postgres")
    cur = conn.cursor()
    cur.execute("""
        SELECT name, entity_version, internal_properties::jsonb->>'format', internal_properties::jsonb->>'location'
        FROM polaris_schema.entities
        WHERE name = %s AND type_code = 7
          AND internal_properties::jsonb->>'parent-namespace' = 'delta_db'
    """, (table,))
    for row in cur.fetchall():
        print(row)
    conn.close()

In [ ]:
show_catalog("mall_customers")

## 4. Append and look again

The table has a new commit in `_delta_log`, but the Polaris row does not change.

In [ ]:
spark.sql("""
    INSERT INTO lakehouse.delta_db.mall_customers
    SELECT * FROM lakehouse.delta_db.mall_customers LIMIT 5
""")
spark.table("lakehouse.delta_db.mall_customers").count()

In [ ]:
show_catalog("mall_customers")

## 5. History and time travel

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{delta_path}`").select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
spark.read.format("delta").option("versionAsOf", 0).load(delta_path).count()

In [ ]:
spark.stop()